# Chapter 04 — Self-Attention from Scratch

Attention is the mechanism that transformed natural language processing forever. Before attention, models processed sequences left-to-right with a fixed context window. Attention allows every position in a sequence to directly 'look at' every other position, enabling the model to capture long-range dependencies that were previously impossible.

In [ ]:
import torch
import torch.nn.functional as F
import math

def self_attention(x, W_q, W_k, W_v, mask=None):
    """
    x: (batch, seq_len, d_model)
    W_q, W_k, W_v: (d_model, d_k)
    """
    Q = x @ W_q  # (batch, seq_len, d_k)
    K = x @ W_k
    V = x @ W_v

    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)

    # Causal mask: prevent attending to future tokens
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))

    attn_weights = F.softmax(scores, dim=-1)
    output = attn_weights @ V  # (batch, seq_len, d_k)
    return output, attn_weights

# Demo: self-attention on random sequence
batch, seq_len, d_model, d_k = 1, 4, 8, 8
x = torch.randn(batch, seq_len, d_model)
W_q = torch.randn(d_model, d_k)
W_k = torch.randn(d_model, d_k)
W_v = torch.randn(d_model, d_k)

# Causal mask (lower triangular)
mask = torch.tril(torch.ones(seq_len, seq_len))
output, attn_weights = self_attention(x, W_q, W_k, W_v, mask)

print(f"Input shape:      {tuple(x.shape)}")
print(f"Output shape:     {tuple(output.shape)}")
print(f"Attention weights (row = query, col = key):")
print(attn_weights[0].detach().numpy().round(3))

---

**Course:** Aprenda LLM | **Chapter 04**

This notebook is part of the [Aprenda LLM](https://magestic.dev) course.